# 🎬 Streaming Content Catalog — EDA & Analysis

**15K titles · 10 platforms · 24 countries · 47 years**

1. Overview
2. Platform Comparison
3. Genre Analysis
4. Rating Distributions
5. Release Year Trends
6. International Content
7. Duration & Format Analysis
8. Budget vs. Ratings
9. Awards & Prestige
10. Content Recommender (Baseline)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print("✅ Ready")

## 1. Overview

In [ ]:
INPUT = "/kaggle/input/streaming-content-catalog-netflix-prime-disney"

df = pd.read_csv(f"{INPUT}/streaming_catalog.csv")
platform_sum = pd.read_csv(f"{INPUT}/platform_summary.csv")
genre_sum = pd.read_csv(f"{INPUT}/genre_summary.csv")
country_sum = pd.read_csv(f"{INPUT}/country_summary.csv")
yearly = pd.read_csv(f"{INPUT}/yearly_release_trends.csv")

print(f"Shape: {df.shape}")
print(f"Platforms: {df['platform'].nunique()} | Countries: {df['country'].nunique()}")
print(f"Year range: {df['release_year'].min()}–{df['release_year'].max()}")
print(f"Avg IMDb: {df['imdb_rating'].mean():.2f}")

In [ ]:
df.head()

## 2. Platform Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

platform_sum.sort_values('total_titles').plot.barh(x='platform', y='total_titles', 
    ax=axes[0,0], color=sns.color_palette("husl", 10), edgecolor='black', legend=False)
axes[0,0].set_title('Titles per Platform', fontsize=13, fontweight='bold')

# Movies vs TV per platform
platform_type = df.groupby(['platform', 'type']).size().unstack(fill_value=0)
platform_type = platform_type.reindex(platform_sum.sort_values('total_titles')['platform'])
platform_type.plot.barh(stacked=True, ax=axes[0,1], color=['#e74c3c', '#3498db'], edgecolor='black')
axes[0,1].set_title('Movies vs TV Shows per Platform', fontsize=13, fontweight='bold')
axes[0,1].legend()

platform_sum.sort_values('avg_imdb').plot.barh(x='platform', y='avg_imdb', 
    ax=axes[1,0], color='#2ecc71', edgecolor='black', legend=False)
axes[1,0].set_title('Average IMDb Rating by Platform', fontsize=13, fontweight='bold')
axes[1,0].set_xlim(6, 7.5)

platform_sum.sort_values('total_hours_watched_million').plot.barh(x='platform', y='total_hours_watched_million',
    ax=axes[1,1], color='#e67e22', edgecolor='black', legend=False)
axes[1,1].set_title('Total Hours Watched (Millions)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Genre Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

genre_counts = df['primary_genre'].value_counts()
genre_counts.sort_values().plot.barh(ax=axes[0], color=sns.color_palette("viridis", len(genre_counts)), edgecolor='black')
axes[0].set_title('Titles per Primary Genre', fontsize=14, fontweight='bold')

genre_imdb = df.groupby('primary_genre')['imdb_rating'].mean().sort_values()
genre_imdb.plot.barh(ax=axes[1], color=sns.color_palette("magma", len(genre_imdb)), edgecolor='black')
axes[1].set_title('Average IMDb by Genre', fontsize=14, fontweight='bold')
axes[1].set_xlim(5.5, 7.5)

plt.tight_layout()
plt.show()

In [ ]:
# Genre mix per platform (top 5 platforms)
top_platforms = df['platform'].value_counts().head(5).index
genre_platform = pd.crosstab(df[df['platform'].isin(top_platforms)]['primary_genre'], 
                              df[df['platform'].isin(top_platforms)]['platform'], normalize='columns') * 100

plt.figure(figsize=(12, 9))
sns.heatmap(genre_platform, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': '% of platform'})
plt.title('Genre Mix per Platform (% of titles)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Rating Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df['imdb_rating'].plot.hist(bins=40, ax=axes[0,0], color='#f39c12', edgecolor='black', alpha=0.8)
axes[0,0].axvline(df['imdb_rating'].mean(), color='red', linestyle='--', label=f"Mean: {df['imdb_rating'].mean():.2f}")
axes[0,0].set_title('IMDb Rating Distribution', fontweight='bold')
axes[0,0].legend()

df['rotten_tomatoes_score'].plot.hist(bins=40, ax=axes[0,1], color='#e74c3c', edgecolor='black', alpha=0.8)
axes[0,1].axvline(df['rotten_tomatoes_score'].mean(), color='blue', linestyle='--', label=f"Mean: {df['rotten_tomatoes_score'].mean():.1f}")
axes[0,1].set_title('Rotten Tomatoes Score Distribution', fontweight='bold')
axes[0,1].legend()

axes[1,0].scatter(df['imdb_rating'], df['rotten_tomatoes_score'], alpha=0.1, s=8, c='#9b59b6')
axes[1,0].set_title('IMDb vs Rotten Tomatoes', fontweight='bold')
axes[1,0].set_xlabel('IMDb Rating')
axes[1,0].set_ylabel('RT Score')

# Movie vs TV ratings
df.boxplot(column='imdb_rating', by='type', ax=axes[1,1])
axes[1,1].set_title('IMDb Rating: Movies vs TV Shows', fontweight='bold')
axes[1,1].set_xlabel('')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 5. Release Year Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].bar(yearly['release_year'], yearly['movies'], label='Movies', color='#e74c3c', alpha=0.8)
axes[0].bar(yearly['release_year'], yearly['tv_shows'], bottom=yearly['movies'], label='TV Shows', color='#3498db', alpha=0.8)
axes[0].set_title('Content Releases per Year (1980–2026)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].plot(yearly['release_year'], yearly['avg_imdb'], marker='o', color='#2ecc71', linewidth=2)
axes[1].set_title('Average IMDb Rating by Release Year', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Avg IMDb')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

## 6. International Content

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_countries = df['country'].value_counts().head(15).sort_values()
top_countries.plot.barh(ax=axes[0], color=sns.color_palette("crest", 15), edgecolor='black')
axes[0].set_title('Top 15 Countries by Title Count', fontsize=14, fontweight='bold')

# Non-US share over time
df['is_non_us'] = df['country'] != 'United States'
non_us_trend = df.groupby('release_year')['is_non_us'].mean() * 100
non_us_trend.plot(ax=axes[1], color='#e67e22', linewidth=2, marker='o')
axes[1].set_title('Non-US Content Share Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('% Non-US')
axes[1].set_xlabel('Release Year')

plt.tight_layout()
plt.show()

## 7. Duration & Format Analysis

In [ ]:
movies = df[df['type'] == 'Movie']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

movies['duration_minutes'].plot.hist(bins=40, ax=axes[0], color='#3498db', edgecolor='black')
axes[0].axvline(movies['duration_minutes'].median(), color='red', linestyle='--', 
                label=f'Median: {movies["duration_minutes"].median():.0f} min')
axes[0].set_title('Movie Duration Distribution', fontweight='bold')
axes[0].set_xlabel('Duration (minutes)')
axes[0].legend()

# Duration by genre
dur_genre = movies.groupby('primary_genre')['duration_minutes'].median().sort_values()
dur_genre.plot.barh(ax=axes[1], color='#2ecc71', edgecolor='black')
axes[1].set_title('Median Movie Duration by Genre', fontweight='bold')
axes[1].set_xlabel('Minutes')

plt.tight_layout()
plt.show()

In [ ]:
# TV show seasons distribution
tv = df[df['type'] == 'TV Show']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
tv['num_seasons'].value_counts().sort_index().plot.bar(ax=axes[0], color='#9b59b6', edgecolor='black')
axes[0].set_title('TV Show Seasons Distribution', fontweight='bold')
axes[0].set_xlabel('Number of Seasons')
axes[0].tick_params(axis='x', rotation=0)

tv['num_episodes'].clip(upper=150).plot.hist(bins=30, ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Total Episodes Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Budget vs. Ratings

In [ ]:
movies = df[df['type'] == 'Movie'].dropna(subset=['budget_million_usd'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(movies['budget_million_usd'], movies['imdb_rating'], alpha=0.15, s=8, c='#e74c3c')
axes[0].set_xscale('log')
axes[0].set_title('Budget vs IMDb Rating', fontweight='bold')
axes[0].set_xlabel('Budget ($M, log scale)')
axes[0].set_ylabel('IMDb Rating')

# Budget bins
budget_bins = pd.cut(movies['budget_million_usd'], 
                     bins=[0, 1, 5, 20, 50, 100, 500],
                     labels=['<$1M', '$1-5M', '$5-20M', '$20-50M', '$50-100M', '$100M+'])
budget_rating = movies.groupby(budget_bins)['imdb_rating'].mean()
budget_rating.plot.bar(ax=axes[1], color=sns.color_palette("YlOrRd", 6), edgecolor='black')
axes[1].set_title('Avg IMDb by Budget Range', fontweight='bold')
axes[1].set_ylabel('Avg IMDb')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 9. Awards & Prestige

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

awards_by_platform = df.groupby('platform')['awards_won'].sum().sort_values()
awards_by_platform.plot.barh(ax=axes[0], color='#f1c40f', edgecolor='black')
axes[0].set_title('Total Awards by Platform', fontweight='bold')

awards_by_genre = df.groupby('primary_genre')['awards_won'].sum().sort_values().tail(10)
awards_by_genre.plot.barh(ax=axes[1], color='#d35400', edgecolor='black')
axes[1].set_title('Top 10 Genres by Total Awards', fontweight='bold')

plt.tight_layout()
plt.show()

# Top awarded titles
top_award = df.nlargest(15, 'awards_won')[['title', 'platform', 'primary_genre', 'imdb_rating', 'awards_won']]
print("\n🏆 Top 15 Most Awarded Titles:\n")
print(top_award.to_string(index=False))

## 10. Content Recommender (Baseline)

In [ ]:
# Simple TF-IDF content-based recommender
df['combined_text'] = df['genres'].fillna('') + ' ' + df['description'].fillna('') + ' ' + df['primary_genre']

tfidf = TfidfVectorizer(max_features=3000, stop_words='english')
matrix = tfidf.fit_transform(df['combined_text'])

def recommend(title, n=5):
    idx = df[df['title'] == title].index
    if len(idx) == 0:
        return None
    idx = idx[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()
    top_idx = sims.argsort()[-n-2:-1][::-1]
    top_idx = [i for i in top_idx if i != idx][:n]
    return df.iloc[top_idx][['title', 'platform', 'primary_genre', 'imdb_rating']]

# Example: top 5 titles
sample_title = df.nlargest(1, 'imdb_rating')['title'].values[0]
print(f"🎬 Recommendations similar to '{sample_title}':\n")
recs = recommend(sample_title)
print(recs.to_string(index=False) if recs is not None else "Title not found")

## 📋 Key Takeaways

- **Netflix dominates** with 30% of catalog share, followed by Amazon Prime and Disney+
- **Drama and Comedy** account for ~35% of all content
- **Non-US content share** has risen dramatically since 2015, driven by K-drama and anime boom
- **HBO Max and Apple TV+** focus on prestige — highest average IMDb scores
- **Budget has diminishing returns** on rating above $50M
- **TV shows average longer audience engagement** but movies have higher peak ratings
- **International platforms** like Crunchyroll and JioCinema show clear genre specialization
- **Content releases peaked around 2018–2022** then plateaued

**Next steps:**
- Hybrid recommender with collaborative + content-based signals
- BERT-based genre/mood classifier from descriptions
- Time-to-platform analysis (how fast do theatrical releases hit streaming?)
- Platform switching prediction

---
*If this was useful, please upvote! 🙏*